# 14.3 - Observability & Evaluation

**Module 14 - LLMOps** | Netsetos GenAI Engineering

This notebook builds the LLM eval toolkit as small, runnable models: a span-tree trace, a versioned golden set, a cheap code-assert screen, an LLM-as-judge rubric, a Cohen's-kappa calibration check, a three-part CI eval gate, and a nightly drift monitor. Every cell runs keyless and deterministic - it models the logic in plain Python so you can see the mechanism, not the API call.

Read top to bottom: each code cell has a short **intro above** it and a **step-by-step walkthrough below** it. Run the cell, then check its output against the walkthrough.

## Setup

This lesson runs entirely on deterministic Python - no model calls - so setup is deliberately thin. The commented-out install line is only needed if you later run the illustrative real-judge snippet against the Anthropic API.

In [ ]:
# Install dependencies (if running in Colab or fresh environment)
# Uncomment the next line if needed:
# !pip install anthropic -q

# Reproducibility - the lesson uses seeded random values throughout

**Explanation**

A one-line environment-prep cell: it carries a commented `pip install anthropic` for the optional live-judge example and a note that the lesson uses seeded, deterministic values throughout. Nothing executes.

**How the code works - step by step**
- **Commented `pip install anthropic`** - uncomment only if you want to run the illustrative production judge call in a Colab or fresh environment.
- **Reproducibility comment** - flags that every code cell below is deterministic and keyless, so outputs are stable run to run.

**In one line:** nothing to run - this just documents the (optional) dependency.

**What you'll see:** (no output - environment prep)

## 1 - Traces vs logs: find where the time actually went

An LLM call is not one API request - it is a small pipeline: load prompt, embed, retrieve, build messages, call the model, parse, log. A flat log gives you one number (the total); a **trace** breaks the request into timed **spans** so you can see which step is the bottleneck. Summing the spans gives the same total, but only the trace tells you *where* to optimise.

In [ ]:
# A trace is the whole request broken into SPANS (sub-operations); a flat log is one line. Summing the spans gives
# the same total a log would, but ONLY the trace tells you WHERE the time went - so you optimise the right thing.
spans = [("load_prompt", 3), ("embed_review", 48), ("retrieve_similar", 31), ("build_messages", 2),
         ("anthropic.messages.create", 1140), ("parse_json", 1), ("log_outcome", 22)]   # ms per span (illustrative)
total_ms = sum(ms for _, ms in spans)
slow_name, slow_ms = max(spans, key=lambda s: s[1])
print("classify_review  (total {} ms)".format(total_ms))
for name, ms in spans:
    bar = "#" * round(ms / total_ms * 40)
    print("  {:<26} {:>5} ms  {:>4.0%}  {}".format(name, ms, ms / total_ms, bar))
print()
print("A flat log says only 'latency: {} ms'.".format(total_ms))
print("The trace says the {} span is {:.0%} of it ({} ms); retrieval is only {} ms.".format(slow_name, slow_ms / total_ms, slow_ms, 31))
print("Without spans you might 'optimise' retrieval and save nothing - the model call is where the time actually is.")

# Output:
# classify_review  (total 1247 ms)
#   load_prompt                    3 ms    0%  
#   embed_review                  48 ms    4%  ##
#   retrieve_similar              31 ms    2%  #
#   build_messages                 2 ms    0%  
#   anthropic.messages.create   1140 ms   91%  #####################################
#   parse_json                     1 ms    0%  
#   log_outcome                   22 ms    2%  #
#
# A flat log says only 'latency: 1247 ms'.
# The trace says the anthropic.messages.create span is 91% of it (1140 ms); retrieval is only 31 ms.
# Without spans you might 'optimise' retrieval and save nothing - the model call is where the time actually is.

**Explanation**

This is a visualization harness, not a real request: a hard-coded list of (span-name, milliseconds) pairs stands in for one traced classification. It sums them into a total, finds the slowest span, and draws an ASCII bar chart so the point is undeniable - the model call is the time, retrieval is noise.

**How the code works - step by step**
- **`spans` list** - seven (name, ms) tuples for one request; the `anthropic.messages.create` span is deliberately the giant one at 1140 ms.
- **`total_ms`** - sums every span's duration; this is the single number a flat log would have shown.
- **`max(spans, key=...)`** - finds the slowest span (the model call) to name the real bottleneck.
- **Bar loop** - for each span, prints its ms, its percent of total, and a `#` bar scaled to 40 chars, so the model call visually dwarfs the rest.
- **Closing prints** - contrast the flat-log view ('latency: 1247 ms') with the trace's insight: the model call is 91% and retrieval only 31 ms.

**In one line:** break the request into spans, sum them, and the bar chart shows the model call is where the time is.

**What you'll see:** A per-span bar chart totalling 1247 ms where `anthropic.messages.create` is 1140 ms (91%) and retrieval only 31 ms, followed by lines explaining that a flat log would have hidden this and you might have wasted effort optimising retrieval.

## 2 - The golden set: split the score by difficulty

The golden set is your most valuable eval artifact: versioned **(input, expected)** pairs re-run on every prompt, model, or retrieval change. The 2026 design has four buckets - a stratified production sample, an adversarial set, constructed edge cases, and replays of shipped failures. The key discipline is including **hard** cases: an easy-only set lets a mediocre prompt score a perfect pass rate while it silently fails everything ambiguous.

In [ ]:
# The golden set is the eval's ground truth: versioned (input, expected) pairs re-run on every change. The 2026
# design has 4 buckets - a stratified production sample, an adversarial set, constructed edge cases, and replays of
# shipped failures. The HARD slice is where you actually learn: an easy-only pass rate hides the real weaknesses.
GOLDEN = [  # (input, expected, difficulty)
    ("pizza cold and late, refund please", "refund", "easy"),
    ("delivery boy was rude", "service", "easy"),
    ("food great but driver took forever", "delivery", "hard"),   # ambiguous: delivery vs service
    ("5 stars, loved it", "other", "easy"),
    ("mi comida estaba fria", "food_quality", "hard"),            # Spanish: "my food was cold"
    ("app crashed when paying", "service", "easy"),
    ("pasta had too much oil", "food_quality", "easy"),
    ("wrong order delivered", "delivery", "easy"),
    ("want to cancel my subscription", "refund", "easy"),
    ("how do I track my order", "other", "easy"),
]
CHAMPION = {  # a basic no-few-shot prompt: misses the two HARD cases (illustrative, deterministic)
    "pizza cold and late, refund please": "refund", "delivery boy was rude": "service",
    "food great but driver took forever": "service", "5 stars, loved it": "other",
    "mi comida estaba fria": "delivery", "app crashed when paying": "service",
    "pasta had too much oil": "food_quality", "wrong order delivered": "delivery",
    "want to cancel my subscription": "refund", "how do I track my order": "other",
}
passes = sum(1 for inp, exp, _ in GOLDEN if CHAMPION[inp] == exp)
easy = [(inp, exp) for inp, exp, d in GOLDEN if d == "easy"]
hard = [(inp, exp) for inp, exp, d in GOLDEN if d == "hard"]
easy_pass = sum(1 for inp, exp in easy if CHAMPION[inp] == exp)
hard_pass = sum(1 for inp, exp in hard if CHAMPION[inp] == exp)
print("Golden set: {} cases. Champion pass-rate: {}/{} = {:.0%}".format(len(GOLDEN), passes, len(GOLDEN), passes / len(GOLDEN)))
print("  easy: {}/{} = {:.0%}    hard: {}/{} = {:.0%}".format(easy_pass, len(easy), easy_pass / len(easy), hard_pass, len(hard), hard_pass / len(hard)))
print("  the champion aces the easy cases and FAILS every hard one - a headline pass-rate would have hidden that.")
print()
print("Re-run this exact set on every prompt/model change; version it in git; grow it by replaying each new production failure.")

# Output:
# Golden set: 10 cases. Champion pass-rate: 8/10 = 80%
#   easy: 8/8 = 100%    hard: 0/2 = 0%
#   the champion aces the easy cases and FAILS every hard one - a headline pass-rate would have hidden that.
#
# Re-run this exact set on every prompt/model change; version it in git; grow it by replaying each new production failure.

**Explanation**

A miniature test-suite runner: `GOLDEN` is ten labelled cases tagged easy or hard, and `CHAMPION` is a fixed lookup that models a basic no-few-shot prompt's answers. It computes the headline pass rate, then re-computes it split by difficulty - and the split is the whole lesson: the champion aces every easy case and misses both hard ones.

**How the code works - step by step**
- **`GOLDEN`** - ten (input, expected, difficulty) tuples; the two `hard` cases are an ambiguous delivery-vs-service review and a Spanish-language one.
- **`CHAMPION`** - a deterministic dict of the prompt's predicted category per input; it deliberately mislabels the two hard cases.
- **`passes`** - counts overall matches between champion and expected.
- **`easy` / `hard` splits** - filter the golden set by difficulty tag, then count passes within each slice.
- **Prints** - show the headline 80% next to the damning split: 100% easy, 0% hard, plus the reminder to version the set in git and grow it by replaying production failures.

**In one line:** re-run the champion on a versioned set, then split by difficulty to expose the weakness the headline number hides.

**What you'll see:** A pass-rate of 8/10 = 80% that looks fine, then the split: easy 8/8 = 100%, hard 0/2 = 0% - the champion aces easy cases and fails every hard one, which the headline number would have hidden.

## 3 - Code asserts: the free first gate before the paid judge

Many failures are **objective** and a plain function catches them: not valid JSON, not an allowed category, too long, leaked an email. These **code asserts** are cheap, instant, and deterministic - run them **first**. An output that fails is rejected for zero cost and zero latency; only what passes is worth sending to the slow, paid LLM judge. This is the metric pyramid: level 3 (objective) before level 4 (subjective).

In [ ]:
# Level 3 (objective) BEFORE level 4 (subjective): cheap, instant, deterministic CODE ASSERTS catch format bugs for
# free - run them FIRST and only send what passes to the slow, paid LLM judge. No API, no cost, no flakiness.
CATEGORIES = {"refund", "delivery", "food_quality", "service", "other"}
import re
def code_asserts(output):
    fails = []
    if output not in CATEGORIES: fails.append("not_in_enum")
    if output != output.lower() or " " in output.strip(): fails.append("not_single_lowercase_token")
    if re.search(r"[\w.+-]+@[\w-]+\.[\w.]+", output): fails.append("contains_email_PII")
    return fails
outputs = ["refund", "REFUND!!! definitely refund", "{'category': 'refund'}",
           "contact me at u@x.com", "delivery", "banana"]
clean = 0
for o in outputs:
    fails = code_asserts(o)
    verdict = "PASS -> send to judge" if not fails else "FAIL (free) -> " + ", ".join(fails)
    if not fails: clean += 1
    print("  {:<32} {}".format(repr(o)[:32], verdict))
print()
print("{}/{} outputs pass the code asserts and are worth a judge call; the rest are rejected for $0 and 0 ms.".format(clean, len(outputs)))
print("Code asserts are deterministic and instant - they are the cheap first gate before the expensive subjective one.")

# Output:
#   'refund'                         PASS -> send to judge
#   'REFUND!!! definitely refund'    FAIL (free) -> not_in_enum, not_single_lowercase_token
#   "{'category': 'refund'}"         FAIL (free) -> not_in_enum, not_single_lowercase_token
#   'contact me at u@x.com'          FAIL (free) -> not_in_enum, not_single_lowercase_token, contains_email_PII
#   'delivery'                       PASS -> send to judge
#   'banana'                         FAIL (free) -> not_in_enum
#
# 2/6 outputs pass the code asserts and are worth a judge call; the rest are rejected for $0 and 0 ms.
# Code asserts are deterministic and instant - they are the cheap first gate before the expensive subjective one.

**Explanation**

A deterministic screening function, not a model call: `code_asserts` runs three regex/string checks - in-enum, single-lowercase-token, no-email-PII - and returns the list of failures. The cell feeds six sample outputs through it and counts how many are even worth a judge call, showing the rejects are caught for free.

**How the code works - step by step**
- **`CATEGORIES` set** - the five allowed labels; the enum membership check keys off this.
- **`code_asserts(output)`** - runs three checks and collects named failures: not in the enum, not a single lowercase token (uppercase or embedded spaces), or contains an email via regex.
- **`outputs` list** - six mixed strings: two clean labels and four malformed (shouting, JSON blob, PII, a nonsense word).
- **Loop** - for each output, marks it PASS -> send to judge, or FAIL (free) with the specific asserts it tripped, and tallies the clean ones.
- **Closing prints** - report that only 2/6 are worth a judge call and the rest were rejected for $0 and 0 ms.

**In one line:** run cheap deterministic checks first so the expensive judge only ever sees outputs that already clear the mechanical bar.

**What you'll see:** Each of six outputs stamped PASS or FAIL with its failed asserts named (e.g. the PII string trips all three: not_in_enum, not_single_lowercase_token, contains_email_PII), ending with '2/6 outputs pass ... the rest are rejected for $0 and 0 ms.'

## 4 - LLM-as-judge: grade freeform output against a rubric

For freeform outputs there is no label to match, so a second, stronger model grades the first against a **rubric**. Four rules make it trustworthy: multiple dimensions (not one blurry score), a small discrete scale (0-2, never a free 1-10 that clusters at 7), the reference answer in the prompt, and pass = all dimensions at least 1. The judge should be a stronger, **different-family** model, because a model over-rewards its own style.

> **The real version needs an Anthropic API key** - the following markdown cell (`llm_judge_real.py`) shows an Opus-class judge call but is not run here.

In [ ]:
# Level 4 (subjective): when the output is freeform, an LLM-AS-JUDGE grades it against a RUBRIC. Rules that make it
# work: multi-dimension, a SMALL discrete scale (0-2, never free 1-10), ground truth in the prompt, pass = all dims >= 1.
def judge_rubric(expected, output):   # models the judge's rubric logic deterministically (real judge = a stronger model)
    correctness = 2 if output == expected else 0
    fmt = 2 if (output == output.lower() and " " not in output.strip()) else 1
    calibration = 2                    # no hedging in a one-word label
    passed = correctness >= 1 and fmt >= 1 and calibration >= 1
    return {"correctness": correctness, "format": fmt, "calibration": calibration, "pass": passed}
cases = [("refund", "refund"), ("delivery", "service"), ("other", "other"),
         ("food_quality", "Food quality."), ("refund", "refund")]
passes = 0
for expected, output in cases:
    v = judge_rubric(expected, output)
    if v["pass"]: passes += 1
    print("  expected={:<12} output={:<16} -> correctness={} format={} calib={}  {}".format(
          expected, repr(output)[:16], v["correctness"], v["format"], v["calibration"], "PASS" if v["pass"] else "FAIL"))
print()
print("Judge pass-rate on these {} cases: {}/{}. A discrete rubric gives a structured verdict, not a vague '7/10'.".format(len(cases), passes, len(cases)))
print("Use a stronger, DIFFERENT-family model as judge, and put the expected/reference answer in the prompt. Then calibrate it.")

# Output:
#   expected=refund       output='refund'         -> correctness=2 format=2 calib=2  PASS
#   expected=delivery     output='service'        -> correctness=0 format=2 calib=2  FAIL
#   expected=other        output='other'          -> correctness=2 format=2 calib=2  PASS
#   expected=food_quality output='Food quality.'  -> correctness=0 format=1 calib=2  FAIL
#   expected=refund       output='refund'         -> correctness=2 format=2 calib=2  PASS
#
# Judge pass-rate on these 5 cases: 3/5. A discrete rubric gives a structured verdict, not a vague '7/10'.
# Use a stronger, DIFFERENT-family model as judge, and put the expected/reference answer in the prompt. Then calibrate it.

**Explanation**

This models the judge's rubric logic in plain Python so it runs keyless and deterministic - `judge_rubric` scores correctness, format, and calibration on a 0-2 scale and derives a pass from all-dimensions-at-least-1. It is a stand-in for the real judge (a stronger model), showing the *shape* of a structured verdict versus a vague single number.

**How the code works - step by step**
- **`judge_rubric(expected, output)`** - scores three dimensions: correctness (2 if it matches expected else 0), format (2 if single lowercase token else 1), calibration (fixed 2 - no hedging in a one-word label), then pass = all dims >= 1.
- **`cases` list** - five (expected, output) pairs including a correct label, a wrong label, and a badly-formatted 'Food quality.'
- **Loop** - runs each case through the rubric, prints the per-dimension scores and PASS/FAIL, and tallies passes.
- **Closing prints** - report 3/5 and stress the real practice: use a stronger, different-family judge, put the reference answer in the prompt, then calibrate it.

**In one line:** score each output on a small discrete multi-dimension rubric so you see *which* dimension failed, not a vague 7/10.

**What you'll see:** Five cases with per-dimension scores and verdicts (e.g. output='service' vs expected='delivery' scores correctness=0 -> FAIL; the misformatted 'Food quality.' scores format=1, correctness=0 -> FAIL), ending 'Judge pass-rate on these 5 cases: 3/5.'

## 5 - Judge calibration: trust the judge only after it agrees with humans

The judge is itself a model, so 'is the judge right?' is a real question. You answer it by calibration: hand-label a sample and measure agreement. The trap is raw agreement, which flatters - two labellers agree by chance a large fraction of the time. **Cohen's kappa** subtracts that chance agreement (bar ~0.6). Two biases also need controlling: side-by-side judges favour the first option (position bias) and their own family (self-preference).

In [ ]:
# You do NOT trust a judge until it agrees with HUMANS. Raw agreement flatters; Cohen's kappa corrects for chance
# agreement (bar ~0.6). And a side-by-side judge favours the FIRST option (position bias) - so average both orderings.
human = [1, 1, 0, 1, 1, 0, 1, 1, 0, 1, 1, 1]   # 12 calibration cases, human pass/fail (the ground truth)
judge = [1, 1, 0, 1, 0, 0, 1, 1, 1, 1, 1, 1]   # the judge disagrees on 2 of them
n = len(human)
agree = sum(1 for h, j in zip(human, judge) if h == j)
po = agree / n
p_h1, p_j1 = sum(human) / n, sum(judge) / n
pe = p_h1 * p_j1 + (1 - p_h1) * (1 - p_j1)      # expected agreement by chance
kappa = (po - pe) / (1 - pe)
print("Calibration ({} cases): raw agreement {}/{} = {:.0%}   Cohen's kappa = {:.2f}".format(n, agree, n, po, kappa))
verdict = "TRUSTWORTHY (kappa >= 0.6)" if kappa >= 0.6 else "NOT YET (kappa < 0.6) - fix the rubric before you trust the scores"
print("  -> {}".format(verdict))
print("  {:.0%} agreement LOOKS fine, but kappa {:.2f} shows it is only borderline once you subtract chance agreement.".format(po, kappa))
print()
first_slot_wins, pairs = 13, 20                 # same 2 outputs, judged A-then-B and B-then-A (illustrative)
print("Position bias: the judge picks the FIRST slot {}/{} = {:.0%} of the time (>55% = biased).".format(first_slot_wins, pairs, first_slot_wins / pairs))
print("Mitigate: run BOTH orderings and average; use a different model family; self-preference bias is a ~5-7% own-family boost.")

# Output:
# Calibration (12 cases): raw agreement 10/12 = 83%   Cohen's kappa = 0.56
#   -> NOT YET (kappa < 0.6) - fix the rubric before you trust the scores
#   83% agreement LOOKS fine, but kappa 0.56 shows it is only borderline once you subtract chance agreement.
#
# Position bias: the judge picks the FIRST slot 13/20 = 65% of the time (>55% = biased).
# Mitigate: run BOTH orderings and average; use a different model family; self-preference bias is a ~5-7% own-family boost.

**Explanation**

A statistics harness, not a model call: two lists (human labels and judge labels over 12 cases) drive a from-scratch Cohen's kappa computation, showing that a healthy-looking 83% agreement collapses to a borderline 0.56 kappa once chance is removed. It then reports an illustrative position-bias figure to make the second failure mode concrete.

**How the code works - step by step**
- **`human` / `judge` lists** - 12 pass/fail labels each; the judge disagrees on exactly 2, treating `human` as ground truth.
- **`po`** - observed agreement: matches divided by n (10/12 = 83%).
- **`pe`** - expected-by-chance agreement, from each labeller's marginal rate of 1s and 0s.
- **`kappa`** - `(po - pe) / (1 - pe)`, the chance-corrected agreement.
- **Verdict print** - compares kappa to the 0.6 bar and flags it as NOT YET trustworthy, noting 83% looks fine but is only borderline once chance is subtracted.
- **Position-bias block** - an illustrative 13/20 = 65% first-slot win rate (>55% = biased), with the mitigation: average both orderings, use a different family, and beware ~5-7% self-preference.

**In one line:** compute kappa (not raw agreement) against humans, and check position bias, before you trust any judge score.

**What you'll see:** Raw agreement 10/12 = 83% but Cohen's kappa = 0.56, verdict 'NOT YET (kappa < 0.6) - fix the rubric', then a position-bias line showing the judge picks the first slot 65% of the time with mitigation advice.

## 6 - The CI eval gate: block bad merges on three independent checks

An eval you run by hand is one you forget. The **CI eval gate** runs the golden set on every PR that touches a prompt, model, or retrieval config, posts the delta, and exits non-zero to block a bad merge. It checks three things, because a change can be bad three ways: an absolute pass-rate **floor**, a max **regression** versus main, and a **cost ceiling**. The ~2pp regression limit is grounded in the significance math of 14.2.

In [ ]:
# The CI EVAL GATE runs the golden set on every PR and BLOCKS the merge unless three checks pass: an absolute pass-rate
# FLOOR, a max REGRESSION vs main, and a COST ceiling. It exits non-zero to fail the pull request. (Significance math: 14.2.)
def eval_gate(base_pass, pr_pass, base_cost, pr_cost, floor=0.92, max_drop=0.02, max_cost_incr=0.30):
    reasons = []
    if pr_pass < floor: reasons.append("below floor {:.0%} (got {:.0%})".format(floor, pr_pass))
    if base_pass - pr_pass > max_drop: reasons.append("regressed {:.0f}pp vs main (max {:.0f}pp)".format((base_pass - pr_pass) * 100, max_drop * 100))
    if (pr_cost - base_cost) / base_cost > max_cost_incr: reasons.append("cost +{:.0%} vs main (max +{:.0%})".format((pr_cost - base_cost) / base_cost, max_cost_incr))
    return ("PASS - merge allowed", reasons) if not reasons else ("BLOCK - merge stopped", reasons)
scenarios = [("clean improvement", 0.92, 0.96, 0.0016, 0.0016),
             ("quality regression", 0.92, 0.89, 0.0016, 0.0016),
             ("too expensive",      0.92, 0.95, 0.0016, 0.0024)]
for name, bp, pp, bc, pc in scenarios:
    decision, reasons = eval_gate(bp, pp, bc, pc)
    print("  {:<20} base {:.0%} -> pr {:.0%}, cost ${:.4f} -> ${:.4f}  ::  {}".format(name, bp, pp, bc, pc, decision))
    for r in reasons:
        print("        - {}".format(r))
print()
print("Every prompt/model/retrieval PR triggers the gate; a green eval suite is the ship gate (eval-driven development).")
print("Thresholds are not arbitrary: ~2pp is about the smallest lift a 200-case golden set can even detect (Lesson 14.2).")

# Output:
#   clean improvement    base 92% -> pr 96%, cost $0.0016 -> $0.0016  ::  PASS - merge allowed
#   quality regression   base 92% -> pr 89%, cost $0.0016 -> $0.0016  ::  BLOCK - merge stopped
#         - below floor 92% (got 89%)
#         - regressed 3pp vs main (max 2pp)
#   too expensive        base 92% -> pr 95%, cost $0.0016 -> $0.0024  ::  BLOCK - merge stopped
#         - cost +50% vs main (max +30%)
#
# Every prompt/model/retrieval PR triggers the gate; a green eval suite is the ship gate (eval-driven development).
# Thresholds are not arbitrary: ~2pp is about the smallest lift a 200-case golden set can even detect (Lesson 14.2).

**Explanation**

A decision function modelling the merge gate: `eval_gate` takes base and PR pass-rates and costs, applies three thresholds, and returns PASS or BLOCK with named reasons. The cell runs three PR scenarios through it - a clean win, a quality regression, and a cost blow-up - showing that any single failing check stops the merge.

**How the code works - step by step**
- **`eval_gate(...)`** - checks three conditions against defaults (floor 0.92, max_drop 0.02, max_cost_incr 0.30): below floor, regressed more than 2pp vs main, or cost up more than 30%; collects each failure as a reason.
- **Return** - PASS with an empty reason list only if all three clear; otherwise BLOCK plus the specific reasons.
- **`scenarios` list** - three PRs: a clean improvement (92%->96%), a quality regression (92%->89%), and a too-expensive change (cost $0.0016->$0.0024).
- **Loop** - runs each through the gate and prints the decision plus any blocking reasons indented beneath.
- **Closing prints** - note every prompt/model/retrieval PR triggers the gate and that the ~2pp threshold is the smallest lift a 200-case set can detect (Lesson 14.2).

**In one line:** three independent checks - floor, regression, cost - and a single failure blocks the merge.

**What you'll see:** Three verdicts: the clean improvement passes; the regression BLOCKs (below floor 92%, regressed 3pp); the expensive PR BLOCKs (cost +50% vs max +30%) - each block lists the exact failing check.

## 7 - Drift detection: catch quality falling with no code change

The CI gate protects you from bad *changes*. Drift is the failure with **no change at all** - your code is untouched but quality falls, usually from a silent provider model update (providers ship far more often than you release), a stale RAG index, or aging prompts. The defence is continuous eval: run the golden set **nightly** against the production champion and alert on a >2pp week-over-week drop. Four signals matter: pass-rate, refusal rate, format-compliance, cost per request.

In [ ]:
# Drift: nothing in YOUR code changed, but quality falls. Run the golden set NIGHTLY against the production champion and
# alert on a >2pp week-over-week drop - the usual culprit is a SILENT provider model update (they ship faster than you release).
nightly = [0.95, 0.95, 0.94, 0.95, 0.93, 0.90, 0.89, 0.89]   # champion pass-rate per night, no prompt change (illustrative)
week_start = nightly[0]
ALERT_DROP = 0.02
print("day  pass-rate  vs week-start  status")
for day, rate in enumerate(nightly):
    drop = week_start - rate
    status = "ALERT (drop > {:.0f}pp)".format(ALERT_DROP * 100) if drop > ALERT_DROP else "ok"
    print("  {:>2}    {:.0%}        {:+.0f}pp         {}".format(day, rate, -drop * 100, status))
print()
print("The champion fell from {:.0%} to {:.0%} ({:+.0f}pp) over the week with ZERO code changes -> suspect a provider model update.".format(
      week_start, nightly[-1], -(week_start - nightly[-1]) * 100))
print("Four drift signals to alert on: golden-set pass-rate, refusal rate, format-compliance rate, cost per request.")
print("Nightly eval costs a few dollars and catches quality regressing while error rate stays green (the 12.8 4th failure mode).")

# Output:
# day  pass-rate  vs week-start  status
#    0    95%        -0pp         ok
#    1    95%        -0pp         ok
#    2    94%        -1pp         ok
#    3    95%        -0pp         ok
#    4    93%        -2pp         ok
#    5    90%        -5pp         ALERT (drop > 2pp)
#    6    89%        -6pp         ALERT (drop > 2pp)
#    7    89%        -6pp         ALERT (drop > 2pp)
#
# The champion fell from 95% to 89% (-6pp) over the week with ZERO code changes -> suspect a provider model update.
# Four drift signals to alert on: golden-set pass-rate, refusal rate, format-compliance rate, cost per request.
# Nightly eval costs a few dollars and catches quality regressing while error rate stays green (the 12.8 4th failure mode).

**Explanation**

A monitoring harness, not a model call: `nightly` is eight days of the champion's golden-set pass-rate with no prompt change, and the cell compares each night to the week-start baseline, firing an alert once the drop exceeds 2pp. It makes the insidious point that error rate stays green while answers quietly get worse.

**How the code works - step by step**
- **`nightly` list** - eight nightly pass-rates that drift from 0.95 down to 0.89 with zero code changes.
- **`week_start` / `ALERT_DROP`** - the day-0 baseline and the 2pp alert threshold.
- **Loop** - for each night, computes the drop from baseline, marks ALERT if it exceeds 2pp else ok, and prints day, rate, delta, and status.
- **Closing prints** - summarise the -6pp week-long slide as the signature of a provider model update, list the four drift signals, and note nightly eval costs a few dollars and catches regression the error rate never shows (the 12.8 fourth failure mode).

**In one line:** log the golden-set pass-rate nightly and alert on a >2pp weekly drop to catch drift the error rate hides.

**What you'll see:** A day-by-day table where the pass-rate slides 95% -> 89% and days 5-7 flip to ALERT (drop > 2pp), concluding the -6pp fall with zero code changes points to a silent provider model update.

These seven pieces are one loop: trace it (see what happened per step), curate it (a golden set with hard cases), screen it (free code asserts), judge it (a rubric-driven LLM judge), calibrate it (kappa against humans, not raw agreement), gate it (block bad merges on three checks), and watch it (nightly drift alerts). The through-line is that "did the model do a good job?" is unanswerable as one number, so you break it into measurable, automatable pieces. Next, Lesson 14.4 (Error Analysis) decides *what* your judges and gate should even measure by building a failure taxonomy from real traces.